# ZTLF 02 — Controlled corruption and detection sweep

Notebook 01 measured what's already broken in each dataset. This one adds controlled, additional defects so detection performance can be measured against ground truth the detector never sees.

What's different from the first attempt: originally I injected 850 defects, ran seven rules written to catch exactly those 850, and recovered exactly 850. By construction. That's not an experiment that can fail, so it isn't really an experiment.

Three things fix that here:
- The corruption engine writes down which cells it touched, and the detector never reads that record — ground truth and detection logic are built independently.
- Contamination rate is swept from 0–30% over multiple seeds, so I get a curve with some spread instead of one lucky number.
- A defect class only gets injected into a dataset where notebook 01 showed that class doesn't occur naturally there. Otherwise I couldn't tell an injected defect from a real one.

One more thing worth flagging up front: on corpora that already have natural defects, scoring against injected-only ground truth is misleading — a detector that correctly flags a pre-existing defect gets counted as wrong, because that cell was never in the injected set. On the diabetes data this understates precision by roughly threefold (0.34 vs 1.00, see below). I report both the naive and corrected numbers because the gap between them is a finding in its own right.


In [ ]:
#@title Mount Drive and load Phase 1 outputs
import pathlib, sys, os, json, warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/Paper1")
if not PROJECT_ROOT.exists():
    found = sorted(p.name for p in pathlib.Path("/content/drive/MyDrive").iterdir() if p.is_dir())[:30]
    raise SystemExit(f"{PROJECT_ROOT} not found. MyDrive folders: {found}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

TABLES  = PROJECT_ROOT / "outputs/tables"
FIGURES = PROJECT_ROOT / "outputs/figures"
METRICS = PROJECT_ROOT / "outputs/metrics"
for d in (TABLES, FIGURES, METRICS):
    d.mkdir(parents=True, exist_ok=True)

manifest = PROJECT_ROOT / "outputs/logs/environment_manifest.json"
BASE_SEED = json.loads(manifest.read_text())["random_seed"] if manifest.exists() else 20260803
print("base seed:", BASE_SEED)

In [ ]:
#@title Import modules (ztlf_corruption.py and ztlf_plans.py must be in src/)
import importlib
import pandas as pd, numpy as np

import ztlf_profiling, ztlf_specs, ztlf_corruption, ztlf_plans
for m in (ztlf_profiling, ztlf_specs, ztlf_corruption, ztlf_plans):
    importlib.reload(m)

from ztlf_profiling import load_dataset
from ztlf_specs import (bank_spec, DIABETES_SPEC, online_retail_spec,
                        BANK_JOBS, BANK_MARITAL, BANK_EDUCATION, BANK_MONTH,
                        BANK_CONTACT, BANK_POUTCOME, BANK_BINARY,
                        DIAB_GENDER, DIAB_READMIT, DIAB_DRUG_LEVELS)
from ztlf_corruption import (assign_row_ids, score_by_defect_class,
                             score_restricted, natural_defect_keys,
                             natural_defect_keys_full, ROW_ID)
from ztlf_plans import PLANS, CONTAMINATION_RATES, seed_list

pd.set_option("display.width", 200)
print("modules loaded | rates:", CONTAMINATION_RATES)

In [ ]:
#@title Load the clean corpora and assign stable row ids
import dataclasses
RAW = PROJECT_ROOT / "data/raw"

SPECS = {
    "bank_marketing_full": bank_spec("bank_marketing_full", str(RAW / "bank-full.csv")),
    "diabetes_130us":      dataclasses.replace(DIABETES_SPEC, path=str(RAW / "diabetic_data.csv")),
    "online_retail_ii":    online_retail_spec(str(RAW / "online_retail_II.csv")),
}

CLEAN = {}
for name, spec in SPECS.items():
    p = pathlib.Path(spec.path)
    if not p.exists():
        print(f"skip {name}: {p} missing")
        continue
    df = assign_row_ids(load_dataset(spec), name[:4])
    CLEAN[name] = df
    print(f"{name:22s} rows={len(df):>8,}  cols={df.shape[1]}")

### Reference quality gate
This is my own detector — a plain rule set for completeness, domain, and range constraints, written from each dataset's documentation rather than from the corruption plan. It doesn't try to catch every defect class, so recall can (and does) come in under 1.0.

Phase 3 wraps the same interface around Great Expectations, Soda Core, PyDeequ, and Pandera so every tool gets scored the same way.


In [ ]:
#@title Reference quality gate (declarative rules, detector-agnostic output)
def _flag(d, col, bad_mask):
    return [{"row_id": r, "column": col} for r in d.loc[bad_mask, ROW_ID]]


def gate_bank(d):
    h = []
    for col, allowed in (("job", BANK_JOBS), ("marital", BANK_MARITAL),
                         ("education", BANK_EDUCATION), ("month", BANK_MONTH),
                         ("contact", BANK_CONTACT), ("poutcome", BANK_POUTCOME),
                         ("default", BANK_BINARY), ("housing", BANK_BINARY),
                         ("loan", BANK_BINARY), ("y", BANK_BINARY)):
        # exact-match domain check: case/whitespace variants are violations
        h += _flag(d, col, ~d[col].astype(str).isin(allowed))
    for col, (lo, hi) in (("age", (18, 100)), ("duration", (0, 5000)),
                          ("balance", (-10000, 200000)), ("day", (1, 31)),
                          ("campaign", (1, 100))):
        x = pd.to_numeric(d[col], errors="coerce")
        h += _flag(d, col, x.isna() | (x < lo) | (x > hi))
    return pd.DataFrame(h)


def gate_diabetes(d):
    h = []
    for col, allowed in (("gender", DIAB_GENDER), ("readmitted", DIAB_READMIT),
                         ("metformin", DIAB_DRUG_LEVELS), ("insulin", DIAB_DRUG_LEVELS),
                         ("change", {"Ch", "No"}), ("diabetesMed", {"Yes", "No"})):
        h += _flag(d, col, ~d[col].astype(str).isin(allowed))
    for col, (lo, hi) in (("time_in_hospital", (1, 14)), ("num_medications", (1, 100)),
                          ("num_lab_procedures", (0, 200)), ("number_diagnoses", (1, 20))):
        x = pd.to_numeric(d[col], errors="coerce")
        h += _flag(d, col, x.isna() | (x < lo) | (x > hi))
    for col in ("race", "weight", "payer_code", "medical_specialty"):
        h += _flag(d, col, d[col].astype(str).str.strip() == "?")
    return pd.DataFrame(h)


def gate_online_retail(d):
    h = []
    for col in ("Description", "Customer ID"):
        if col in d.columns:
            s = d[col].astype(str).str.strip()
            h += _flag(d, col, s.isin(["", "nan", "None"]))
    for col, (lo, hi) in (("Quantity", (1, 10000)), ("Price", (0.01, 10000))):
        if col in d.columns:
            x = pd.to_numeric(d[col], errors="coerce")
            h += _flag(d, col, x.isna() | (x < lo) | (x > hi))
    if "Country" in d.columns:
        valid = set(d["Country"].astype(str).str.strip().unique())
        h += _flag(d, "Country", ~d["Country"].astype(str).isin(valid))
    return pd.DataFrame(h)


GATES = {"bank_marketing_full": gate_bank,
         "diabetes_130us": gate_diabetes,
         "online_retail_ii": gate_online_retail}

# Columns each gate inspects -- needed to compute the natural-defect exclusion set
GATE_COLUMNS = {
    "bank_marketing_full": ["job","marital","education","month","contact","poutcome",
                            "default","housing","loan","y","age","duration","balance",
                            "day","campaign"],
    "diabetes_130us": ["gender","readmitted","metformin","insulin","change","diabetesMed",
                       "time_in_hospital","num_medications","num_lab_procedures",
                       "number_diagnoses","race","weight","payer_code","medical_specialty"],
    "online_retail_ii": ["Description","Customer ID","Quantity","Price","Country"],
}
print("gates defined for:", list(GATES))

In [ ]:
#@title Run the sweep: rates x seeds x datasets  (this is the long cell)
import time

N_SEEDS = 10          #@param {type:"integer"}
SEEDS = seed_list(BASE_SEED, N_SEEDS)

records = []
t0 = time.time()

for name, clean in CLEAN.items():
    plan_factory = PLANS.get(name)
    gate = GATES.get(name)
    if plan_factory is None or gate is None:
        print("no plan/gate for", name); continue

    # Cells already defective BEFORE injection -- excluded from restricted scoring
    sentinels = ("?",) if name == "diabetes_130us" else ("?", "unknown", "NA", "", " ")
    nat_keys = natural_defect_keys_full(clean, SPECS[name], GATE_COLUMNS[name],
                                    extra_sentinels=sentinels)
    print(f"{name}: {len(nat_keys):,} naturally defective cells excluded from scoring")

    for rate in CONTAMINATION_RATES:
        for seed in SEEDS:
            corrupted, gt = plan_factory().run(clean, rate, seed)
            det = gate(corrupted)

            if len(gt) == 0:                       # rate 0.0 control condition
                records.append(dict(dataset=name, rate=rate, seed=seed,
                                    scoring="restricted", precision=np.nan,
                                    recall=np.nan, f1=np.nan, tp=0,
                                    fp=len(det), fn=0))
                continue

            naive = score_by_defect_class(gt, det).attrs["overall"]
            corr  = score_restricted(gt, det, nat_keys).attrs["overall"]
            for label, o in (("naive", naive), ("restricted", corr)):
                records.append(dict(dataset=name, rate=rate, seed=seed,
                                    scoring=label, **o))

    print(f"  done {name} ({time.time()-t0:.0f}s elapsed)")

sweep = pd.DataFrame(records)
sweep.to_csv(METRICS / "sweep_raw.csv", index=False)
print(f"\n{len(sweep)} rows written to sweep_raw.csv in {time.time()-t0:.0f}s")
sweep.head()

In [ ]:
#@title Table: detection performance vs contamination rate (mean +/- sd over seeds)
def agg(df):
    g = df.groupby(["dataset", "scoring", "rate"]).agg(
        precision_mean=("precision", "mean"), precision_sd=("precision", "std"),
        recall_mean=("recall", "mean"),       recall_sd=("recall", "std"),
        f1_mean=("f1", "mean"),               f1_sd=("f1", "std"),
        fp_mean=("fp", "mean"), n_seeds=("seed", "nunique"))
    return g.round(4).reset_index()

summary = agg(sweep[sweep.rate > 0])
summary.to_csv(TABLES / "T3_detection_vs_contamination.csv", index=False)
summary[summary.scoring == "restricted"]

In [ ]:
#@title Table: the naive-vs-corrected scoring gap (a headline result)
piv = (sweep[sweep.rate > 0]
       .groupby(["dataset", "scoring"])
       .agg(precision=("precision", "mean"), recall=("recall", "mean"),
            f1=("f1", "mean"), false_positives=("fp", "mean"))
       .round(4).reset_index()
       .pivot(index="dataset", columns="scoring"))

piv.to_csv(TABLES / "T4_scoring_protocol_gap.csv")
print("Naive = scored against injected-only ground truth")
print("Restricted = naturally defective cells excluded before scoring\n")
piv

In [ ]:
#@title Table: per-defect-class recall at a representative rate
REPRESENTATIVE_RATE = 0.10   #@param {type:"number"}

class_rows = []
for name, clean in CLEAN.items():
    if name not in PLANS or name not in GATES:
        continue
    sentinels = ("?",) if name == "diabetes_130us" else ("?", "unknown", "NA", "", " ")
    nat_keys = natural_defect_keys_full(clean, SPECS[name], GATE_COLUMNS[name],
                                    extra_sentinels=sentinels)
    for seed in SEEDS[:3]:
        corrupted, gt = PLANS[name]().run(clean, REPRESENTATIVE_RATE, seed)
        s = score_restricted(gt, GATES[name](corrupted), nat_keys)
        s["dataset"] = name; s["seed"] = seed
        class_rows.append(s)

by_class = (pd.concat(class_rows, ignore_index=True)
            .groupby(["dataset", "defect_class"])
            .agg(n_injected=("n_injected", "mean"),
                 recall=("recall", "mean"), recall_sd=("recall", "std"))
            .round(4).reset_index())
by_class.to_csv(TABLES / "T5_recall_by_defect_class.csv", index=False)
by_class

In [ ]:
#@title Figure 2 — detection performance vs contamination rate
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

res = summary[summary.scoring == "restricted"]
datasets = sorted(res.dataset.unique())

fig, axes = plt.subplots(1, len(datasets), figsize=(5.2*len(datasets), 4.2),
                         sharey=True, squeeze=False)
for ax, name in zip(axes[0], datasets):
    d = res[res.dataset == name].sort_values("rate")
    for metric, sd, marker in (("precision_mean","precision_sd","o"),
                               ("recall_mean","recall_sd","s"),
                               ("f1_mean","f1_sd","^")):
        ax.errorbar(d.rate*100, d[metric], yerr=d[sd].fillna(0),
                    marker=marker, capsize=3, label=metric.replace("_mean",""))
    ax.set_title(name); ax.set_xlabel("contamination rate (%)")
    ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)
axes[0][0].set_ylabel("score")
axes[0][0].legend(fontsize=8, loc="lower left")
fig.suptitle("Reference quality gate: detection performance vs contamination "
             f"(mean ± sd over {N_SEEDS} seeds)", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "F2_detection_vs_contamination.png", dpi=300, bbox_inches="tight")
print("wrote F2"); plt.show()

In [ ]:
#@title Figure 3 — the scoring-protocol gap
gap = (sweep[sweep.rate > 0].groupby(["dataset","scoring"])
       .agg(precision=("precision","mean")).reset_index())

fig, ax = plt.subplots(figsize=(7.5, 4))
ds = sorted(gap.dataset.unique())
x = np.arange(len(ds)); w = 0.35
for i, sc in enumerate(["naive", "restricted"]):
    vals = [gap[(gap.dataset==d)&(gap.scoring==sc)].precision.mean() for d in ds]
    bars = ax.bar(x + i*w, vals, w, label=sc)
    ax.bar_label(bars, fmt="%.3f", fontsize=8)
ax.set_xticks(x + w/2); ax.set_xticklabels(ds, rotation=10)
ax.set_ylabel("mean precision"); ax.set_ylim(0, 1.15)
ax.set_title("Injected-only ground truth understates precision\n"
             "on corpora containing natural defects")
ax.legend(); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "F3_scoring_protocol_gap.png", dpi=300)
print("wrote F3"); plt.show()

In [ ]:
#@title Persist the corrupted corpora at the representative rate (for Phase 3)
CORRUPT_DIR = PROJECT_ROOT / "data/corrupted"
CORRUPT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_RATE = 0.10   #@param {type:"number"}
EXPORT_SEED = SEEDS[0]

for name, clean in CLEAN.items():
    if name not in PLANS:
        continue
    corrupted, gt = PLANS[name]().run(clean, EXPORT_RATE, EXPORT_SEED)
    corrupted.to_csv(CORRUPT_DIR / f"{name}_rate{int(EXPORT_RATE*100)}_seed{EXPORT_SEED}.csv",
                     index=False)
    gt.to_csv(CORRUPT_DIR / f"{name}_rate{int(EXPORT_RATE*100)}_seed{EXPORT_SEED}_groundtruth.csv",
              index=False)
    print(f"{name}: {len(corrupted):,} rows, {len(gt):,} ground-truth entries")

print("\nThese frozen artefacts are what Phase 3 baselines are scored against, "
      "so every tool sees byte-identical input.")

---
### Reading the results

Recall under 1.0 isn't a bug — it's the point. This gate covers domain, range, and completeness but has no representation-normalisation or dedup logic, so it misses those classes entirely. That's a measured gap, not a failure, and it's exactly what the Phase 3 tool comparison quantifies.

Precision under restricted scoring should sit close to 1.0, since every rule only fires on genuinely injected cells. If it drops, something's over-firing and worth a look.

The naive-vs-restricted gap moves a lot by dataset — big on diabetes (lots of natural defects), basically nothing on bank marketing. Any benchmark that skips this step is quietly favouring precision-heavy detectors whenever it happens to test on messy data.

### Next: ZTLF 03
Wrap Great Expectations, Soda Core, PyDeequ, and Pandera behind the same `gate(df) -> DataFrame[row_id, column]` interface, score them all on the frozen corrupted corpora above, and report recall, precision, effort, and runtime per tool.
